# Rainbow DQN: understand how six improvements compose

Rainbow combines Double targets, dueling heads, prioritized replay (PER), n-step returns, NoisyNet exploration, and C51 distributional learning:
$$\mathcal L_i=w_i\left[-\sum_j(\Phi T^{(n)}Z)_j\log p_j(S_i,A_i)\right].$$
Here $T^{(n)}$ is a Double-DQN n-step distributional target, $\Phi$ the C51 projection, $p_j$ predicted atom probabilities, and $w_i$ the PER importance weight.

## 1. Map each component to its role

A compact table keeps architecture, target, sampling, exploration, and loss responsibilities separate.

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

components = {
    "Double": "select online, evaluate target",
    "Dueling": "value + centered advantage heads",
    "PER": "sample by TD-error priority",
    "n-step": "accumulate several rewards",
    "NoisyNet": "learn parameter-space noise",
    "C51": "predict probabilities on fixed atoms",
}
for name, role in components.items():
    print(f"{name:9s} -> {role}")

## 2. Compute a combined target

This numerical example follows the data path: n-step reward, Double action selection, categorical shift, then importance weighting.

In [ ]:
GAMMA = 0.99
N_STEPS = 3
ATOMS = 51
V_MIN, V_MAX = -10.0, 10.0
rewards = np.array([1.0, 1.0, 1.0])
n_step_reward = sum(GAMMA**k * reward for k, reward in enumerate(rewards))
online_q = np.array([2.0, 3.0])
target_q = np.array([2.5, 2.7])
selected_action = int(online_q.argmax())
bootstrap = target_q[selected_action]
scalar_target = n_step_reward + GAMMA**N_STEPS * bootstrap
priority = abs(scalar_target - 1.5) + 1e-6
importance_weight = (10 * (priority**0.6 / priority**0.6))**(-0.4)
print("Double-selected action:", selected_action)
print("n-step target:", scalar_target)
print("priority:", priority, "weight:", importance_weight)

## 3. Visualize the categorical shift

C51 clips the shifted atoms to its fixed support before projection.

In [ ]:
support = np.linspace(V_MIN, V_MAX, ATOMS)
shifted = np.clip(n_step_reward + GAMMA**N_STEPS * support, V_MIN, V_MAX)
plt.plot(support, shifted)
plt.xlabel("Original atom")
plt.ylabel("n-step shifted atom")
plt.title("Rainbow distributional target transform")
plt.grid(alpha=0.2); plt.show()